# TP 4 — Spark SQL : requêter le fil rouge e-commerce

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

## Consignes
- Complétez toutes les cellules marquées `# === À COMPLÉTER ===` (remplacez les `...`).
- Rédigez vos réponses dans les cellules *Votre réponse :*.
- Le notebook doit s'exécuter **de bout en bout** (Kernel > Restart & Run All) avant d'être poussé.
- Livrable : `notebooks/TP4_spark_sql.ipynb` **avec les sorties visibles**, poussé sur votre dépôt avant la séance 5.

## Déroulé
| Partie | Contenu | Durée |
|---|---|---|
| A | Mise en place : données, SparkSession, vues | 15 min |
| B | Premières requêtes SQL | 25 min |
| C | L'enquête FCFA | 30 min |
| D | Les indicateurs de la direction | 40 min |
| E | SQL ou API ? `explain()` tranche | 20 min |
| F | Discussion et quiz | 20 min |

## 0. Vérification de l'environnement

Données : si le dossier `data/` est absent, exécutez d'abord dans un terminal (ou une cellule `!`) :
```
python generate_data.py --scale 0.1 --outdir data
```
Graine 42 : tous les étudiants ont **exactement** les mêmes données.

In [32]:
import sys
print("Python :", sys.version.split()[0])

from pyspark.sql import SparkSession
spark = (SparkSession.builder
        .appName("TP4-SparkSQL")
        .master("local[*]")
        .getOrCreate())
print("Spark  :", spark.version)
spark.sparkContext.setLogLevel("ERROR")

Python : 3.13.5
Spark  : 3.5.9


### Tableau de relevés

Il se remplit **au fil du TP** ; la dernière cellule du notebook l'affiche. Un notebook sans chiffres n'est pas un livrable.

In [33]:
releves = {
    "A_nb_lignes_clients":        None,
    "A_nb_lignes_commandes":      None,
    "A_type_montant_total_fcfa":  None,   # ex. "string"
    "C2_nb_valeurs_polluees":     None,
    "C1_ca_naif":                 None,
    "C4_ca_nettoye":              None,
    "C5_ecart_fcfa":              None,
    "C5_ecart_pct":               None,
    "D1_part_ca_livree_pct":      None,
    "D2_mois_record":             None,
    "D3_panier_moyen_mobile":     None,
    "D5_part_mobile_money_pct":   None,
}

## Partie A — Mise en place (15 min)

### A.1 — Charger les quatre sources et créer les vues

Chargez `customers.csv`, `orders.csv`, `products.csv` (CSV : `header=True`, `inferSchema=True`) et `payments.json`, puis créez les vues temporaires `clients`, `commandes`, `produits`, `paiements`.

In [34]:
base = "../data/"

# === À COMPLÉTER ===
clients   = spark.read.csv(base + "customers.csv", header=True, inferSchema=True)
commandes = spark.read.csv(base + "orders.csv", header=True, inferSchema=True)
produits  = spark.read.csv(base + "products.csv", header=True, inferSchema=True)
paiements = spark.read.json(base + "payments.json")

clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")
produits.createOrReplaceTempView("produits")
paiements.createOrReplaceTempView("paiements")

spark.catalog.listTables()


[Table(name='clients', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='commandes', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='commandes_clean', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='paiements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='produits', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

### A.2 — Premier relevé

Comptez les lignes de `clients` et `commandes`, affichez le schéma de `commandes`, et relevez le **type inféré** de `montant_total_fcfa` et de `frais_livraison_fcfa`.

In [35]:
# === À COMPLÉTER ===
releves["A_nb_lignes_clients"]   = spark.sql("SELECT COUNT(*) AS n FROM clients").first()["n"]
releves["A_nb_lignes_commandes"] = spark.sql("SELECT COUNT(*) AS n FROM commandes").first()["n"]

commandes.printSchema()
releves["A_type_montant_total_fcfa"] = commandes.schema["montant_total_fcfa"].dataType.simpleString()
print("Type montant_total_fcfa :", releves["A_type_montant_total_fcfa"])
print("Type frais_livraison_fcfa :", commandes.schema["frais_livraison_fcfa"].dataType.simpleString())
print(releves)


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

Type montant_total_fcfa : string
Type frais_livraison_fcfa : int
{'A_nb_lignes_clients': 5025, 'A_nb_lignes_commandes': 50000, 'A_type_montant_total_fcfa': 'string', 'C2_nb_valeurs_polluees': None, 'C1_ca_naif': None, 'C4_ca_nettoye': None, 'C5_ecart_fcfa': None, 'C5_ecart_pct': None, 'D1_part_ca_livree_pct': None, 'D2_mois_record': None, 'D3_panier_moyen_mobile': None, 'D5_part_mobile_money_pct': None}


**Question A** — Une des deux colonnes de montants n'a pas le type attendu. Laquelle, et qu'en déduisez-vous sur le contenu du fichier ? (Vous vérifierez votre hypothèse en partie C.)

**Réponse :** 
La colonne montant_total_fcfa n’a pas le type attendu : elle est inférée en string, contrairement à frais_livraison_fcfa qui est en int. Cela suggère que le fichier contient des valeurs mal formatées ou polluées dans cette colonne.

## Partie B — Premières requêtes SQL (25 min)

Une requête par cellule, résultat affiché avec `.show()`.

### B1 — Les 10 premiers clients de Dakar
Colonnes : `customer_id`, `prenom`, `nom`, `ville`.

In [36]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT customer_id, prenom, nom, ville
    FROM clients
    WHERE ville = 'Dakar'
    LIMIT 10
""").show()


+-----------+--------+--------+-----+
|customer_id|  prenom|     nom|ville|
+-----------+--------+--------+-----+
|    C000878|  Yacine|    Faye|Dakar|
|    C002485|   Astou|  Ndiaye|Dakar|
|    C000812|  Diarra|    Wade|Dakar|
|    C000249|Seynabou|Goudiaby|Dakar|
|    C003299|   Adama|    Fall|Dakar|
|    C001282|  Yacine|      Sy|Dakar|
|    C000334|Maguette|   Mendy|Dakar|
|    C002548| Rokhaya|   Badji|Dakar|
|    C002842|    Omar|  Diallo|Dakar|
|    C001159|  Sokhna|   Dieng|Dakar|
+-----------+--------+--------+-----+



### B2 — Combien de villes distinctes dans `clients` ?

In [37]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT COUNT(DISTINCT ville) AS nb_villes
    FROM clients
""").show()


+---------+
|nb_villes|
+---------+
|       56|
+---------+



### B3 — Top 10 des produits les plus chers
Nom et prix, tri décroissant.

In [38]:
spark.sql("""
    SELECT nom_produit, prix_unitaire_fcfa
    FROM produits
    ORDER BY prix_unitaire_fcfa DESC
    LIMIT 10
""").show(truncate=False)

+---------------------------+------------------+
|nom_produit                |prix_unitaire_fcfa|
+---------------------------+------------------+
|Hisense Informatique 0593  |844000            |
|Royal Informatique 0439    |823500            |
|LG Informatique 0469       |813500            |
|Kirène Informatique 0577   |707000            |
|Adidas Informatique 0383   |698000            |
|HP Électroménager 0377     |587500            |
|Sunu Tech Informatique 0340|587000            |
|Sunu Tech Informatique 0180|586000            |
|Infinix Informatique 0445  |584000            |
|Samsung Électroménager 0063|571000            |
+---------------------------+------------------+



### B4 — Commandes livrées, canal mobile, décembre 2025
Combien de commandes `livree` du canal `mobile_app` en décembre 2025 ?

In [39]:
spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE statut = 'livree'
      AND canal = 'mobile_app'
      AND date_commande BETWEEN '2025-12-01' AND '2025-12-31'
""").show()

+---+
| nb|
+---+
|  0|
+---+



### B5 — Emails manquants
Combien de clients ont un email `NULL` **ou** égal à `'N/A'` ? (Rappel séance 3 : le manquant a deux visages — et `= NULL` ne fonctionne pas.)

In [40]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT COUNT(*) AS nb
    FROM clients
    WHERE email IS NULL OR email = 'N/A'
""").show()


+---+
| nb|
+---+
|150|
+---+



## Partie C — L'enquête FCFA (30 min)

### C1 — Le symptôme : la somme naïve
Calculez le CA total directement sur la colonne brute, et **notez le résultat**.

In [41]:
# === À COMPLÉTER ===
ca_naif = spark.sql("""
    SELECT SUM(montant_total_fcfa) AS ca FROM commandes
""").first()["ca"]
releves["C1_ca_naif"] = ca_naif
print(f"CA naif : {ca_naif:,.0f}")

CA naif : 11,519,493,000


### C2 — Diagnostiquer
Comptez les valeurs de `montant_total_fcfa` qui ne sont **pas** de purs nombres, puis affichez 10 valeurs fautives distinctes.

In [42]:
# === À COMPLÉTER ===
nb_pollues = spark.sql("""
    SELECT COUNT(*) AS nb
    FROM commandes
    WHERE TRIM(montant_total_fcfa) NOT RLIKE '^[0-9]+$'
""").first()["nb"]
releves["C2_nb_valeurs_polluees"] = nb_pollues
print("valeurs polluees :", nb_pollues)

spark.sql("""
    SELECT DISTINCT montant_total_fcfa
    FROM commandes
    WHERE TRIM(montant_total_fcfa) NOT RLIKE '^[0-9]+$'
    LIMIT 10
""").show(truncate=False)


valeurs polluees : 500
+------------------+
|montant_total_fcfa|
+------------------+
|845500 FCFA       |
|554000 FCFA       |
|87700 FCFA        |
|21000 FCFA        |
|245500 FCFA       |
|154000 FCFA       |
|187000 FCFA       |
|57000 FCFA        |
|56700 FCFA        |
|1123900 FCFA      |
+------------------+



**Question C** — Expliquez en deux phrases pourquoi la requête C1 rend un résultat **faux sans lever d'erreur**.

*Réponse :*

La requête C1 donne un résultat faux sans lever d’erreur parce que la colonne montant_total_fcfa contient des valeurs brutes et mal formatées, parfois non numériques, et Spark tente de les agréger comme des données textuelles au lieu de les nettoyer avant le calcul. Le résultat est donc trompeur, car il ne représente pas le vrai CA, mais un total calculé sur des valeurs polluées.

### C3 — Nettoyer : la vue `commandes_clean`
Complétez la regex : supprimer **tout ce qui n'est pas un chiffre**, puis caster en `BIGINT`. On conserve la colonne brute sous `montant_raw`.

In [43]:
# === À COMPLÉTER ===
spark.sql("""
    CREATE OR REPLACE TEMP VIEW commandes_clean AS
    SELECT order_id, customer_id, date_commande, statut, canal,
        frais_livraison_fcfa,
        montant_total_fcfa AS montant_raw,
        CAST(regexp_replace(trim(montant_total_fcfa),
            '[^0-9]', '') AS BIGINT) AS montant_fcfa
    FROM commandes
""")
spark.sql("SELECT montant_raw, montant_fcfa FROM commandes_clean LIMIT 5").show()

+-----------+------------+
|montant_raw|montant_fcfa|
+-----------+------------+
|     190500|      190500|
|      32200|       32200|
|      40500|       40500|
|       4000|        4000|
|      35500|       35500|
+-----------+------------+



### C4 — Valider : mesurer, pas affirmer
Vérifiez qu'aucun `NULL` n'a été produit, contrôlez `MIN`/`MAX`, et recalculez le CA.

In [44]:
# === À COMPLÉTER ===
validation = spark.sql("""
    SELECT COUNT(*)                        AS nb_lignes,
           COUNT(montant_fcfa)             AS nb_castes,
           COUNT(*) - COUNT(montant_fcfa)  AS nb_null,
           MIN(montant_fcfa)               AS mini,
           MAX(montant_fcfa)               AS maxi,
           SUM(montant_fcfa)               AS ca_total
    FROM commandes_clean
""")
validation.show()
releves["C4_ca_nettoye"] = int(validation.first()["ca_total"])
print("CA nettoyé :", releves["C4_ca_nettoye"])

+---------+---------+-------+----+-------+-----------+
|nb_lignes|nb_castes|nb_null|mini|   maxi|   ca_total|
+---------+---------+-------+----+-------+-----------+
|    50000|    50000|      0| 500|4182000|11645231000|
+---------+---------+-------+----+-------+-----------+

CA nettoyé : 11645231000


### C5 — La preuve chiffrée
Calculez l'écart entre le CA naïf (C1) et le CA nettoyé (C4), en FCFA et en pourcentage.

In [45]:
# === À COMPLÉTER ===
ecart = releves["C4_ca_nettoye"] - releves["C1_ca_naif"]
releves["C5_ecart_fcfa"] = ecart
releves["C5_ecart_pct"]  = 100 * ecart / releves["C4_ca_nettoye"]
print(f"Ecart : {ecart:,.0f} FCFA soit {releves['C5_ecart_pct']:.2f} %")


Ecart : 125,738,000 FCFA soit 1.08 %


## Partie D — Les indicateurs de la direction (40 min)

Toutes les requêtes portent sur `commandes_clean` (et `paiements` pour D5). Après chaque résultat, ajoutez **une phrase d'interprétation métier** dans la cellule markdown qui suit.

### D1 — CA et commandes par statut
Quelle part du CA est réellement `livree` ?

In [47]:
releves["C4_ca_nettoye"] = validation.first()["ca_total"]
print(releves["C4_ca_nettoye"])

11645231000


In [ ]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT statut,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY statut
    ORDER BY ca_fcfa DESC
""").show()

ca_nettoye = spark.sql("""
    SELECT SUM(montant_fcfa) AS ca_total
    FROM commandes_clean
""").first()["ca_total"]

if releves["C4_ca_nettoye"] is None:
    releves["C4_ca_nettoye"] = int(ca_nettoye)

ca_livre = spark.sql("""
    SELECT SUM(montant_fcfa) AS ca
    FROM commandes_clean
    WHERE statut = 'livree'
""").first()["ca"]

releves["D1_part_ca_livree_pct"] = 100 * ca_livre / releves["C4_ca_nettoye"]
print(f"Part du CA livrée : {releves['D1_part_ca_livree_pct']:.1f} %")

+---------+------------+----------+
|   statut|nb_commandes|   ca_fcfa|
+---------+------------+----------+
|   livrée|       38890|9090757800|
|  annulée|        4568|1066017300|
| en_cours|        4058| 911567000|
|retournée|        2484| 576888900|
+---------+------------+----------+



ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'releves'.(line 7, pos 27)

== SQL ==


    SELECT SUM(montant_fcfa) AS ca

    FROM commandes_cleanprint(f"Part du CA livrée : {releves['D1_part_ca_livree_pct']:.1f} %")

    WHERE statut = 'livree'releves["D1_part_ca_livree_pct"] = 100 * ca_livre / releves["C4_ca_nettoye"]
---------------------------^^^


*Votre réponse :*

…

### D2 — Le CA mensuel des commandes livrées
`date_trunc('month', ...)`, tri chronologique. Repérez la tendance et le mois record.

In [ ]:
# === À COMPLÉTER ===
ca_mensuel = spark.sql("""
    SELECT date_trunc('month', date_commande) AS mois,
           COUNT(*)                           AS nb_commandes,
           SUM(montant_fcfa)                  AS ca_fcfa
    FROM commandes_clean
    WHERE ...
    GROUP BY ...
    ORDER BY mois
""")
ca_mensuel.show(24, truncate=False)
releves["D2_mois_record"] = ...

*Votre réponse :*

…

### D3 — Panier moyen par canal
`ROUND(AVG(montant_fcfa), 0)` — mobile ou web, qui dépense le plus par commande ?

In [ ]:
# === À COMPLÉTER ===
panier = spark.sql("""
    SELECT canal,
           COUNT(*)                    AS nb_commandes,
           SUM(montant_fcfa)           AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY ...
    ORDER BY ca_fcfa DESC
""")
panier.show()
releves["D3_panier_moyen_mobile"] = ...

*Votre réponse :*

…

### D4 — Top clients, avec HAVING
Top 10 des clients par CA (`GROUP BY customer_id`), en ne gardant que les clients dépassant **1 000 000 FCFA** de CA cumulé. Un client sort-il du lot ?

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT customer_id,
           COUNT(*)          AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa
    FROM commandes_clean
    GROUP BY customer_id
    HAVING ...
    ORDER BY ca_fcfa DESC
    LIMIT 10
""").show()

*Votre réponse :*

…

### D5 — Paiements par méthode
Nombre et pourcentage par méthode. Quelle part totale pour le **mobile money** (Orange Money + Wave + Free Money) ?

In [ ]:
# === À COMPLÉTER ===
spark.sql("""
    SELECT methode,
           COUNT(*) AS nb,
           ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS part_pct
    FROM paiements
    GROUP BY methode
    ORDER BY nb DESC
""").show()

releves["D5_part_mobile_money_pct"] = ...

*Votre réponse :*

…

## Partie E — SQL ou API ? `explain()` tranche (20 min)

### E1 — D3 en API DataFrame
Réécrivez le panier moyen par canal avec `groupBy().agg()`. Les chiffres doivent être **identiques**.

In [ ]:
from pyspark.sql import functions as F

# === À COMPLÉTER ===
commandes_clean_df = spark.table("commandes_clean")

panier_api = (commandes_clean_df
    .groupBy("canal")
    .agg(...)
    .orderBy(F.col("ca_fcfa").desc()))
panier_api.show()

### E2 — B4 en API
La même requête « livrées / mobile / décembre 2025 », version `filter`.

In [ ]:
# === À COMPLÉTER ===
nb_api = (spark.table("commandes")
    .filter(...)
    .count())
print(nb_api)

### E3 — Comparer les plans
Affichez le plan physique de la version SQL de D3 et de `panier_api`.

In [ ]:
panier_sql = spark.sql("""
    SELECT canal, COUNT(*) AS nb_commandes,
           SUM(montant_fcfa) AS ca_fcfa,
           ROUND(AVG(montant_fcfa), 0) AS panier_moyen_fcfa
    FROM commandes_clean
    GROUP BY canal
    ORDER BY ca_fcfa DESC
""")
panier_sql.explain()
panier_api.explain()

**Question E** — Les plans physiques sont-ils identiques ? Où voit-on le filtre poussé vers la lecture (`PushedFilters` / `Filter` près du `FileScan`) ? Que concluez-vous sur le choix SQL vs API ? (3 phrases)

*Votre réponse :*

…

## Partie F — Discussion guidée (10 min, en groupes)

1. Le CA naïf de C1 était faux **en silence**. Dans une vraie entreprise, qui s'en serait aperçu, quand, et à quel coût ? Proposez **deux garde-fous techniques**.
2. `commandes_clean` est une vue temporaire : que se passe-t-il demain matin au redémarrage du notebook ? Est-ce acceptable en production ? (indice : séances 6-7)
3. La direction veut le CA par **ville du client** : quelle information manque à `commandes_clean` seule, et comment l'obtiendrez-vous en séance 5 ?

*Votre réponse :*

…

## Quiz éclair (10 min)

1. Que retourne `spark.sql(...)` : une liste, un DataFrame ou un fichier ?
2. `createOrReplaceTempView` copie-t-elle les données ? Quelle est la portée de la vue ?
3. Pourquoi `WHERE email = NULL` ne renvoie-t-il jamais rien ?
4. `SUM` sur une colonne `string` polluée : erreur ou résultat faux ? Pourquoi est-ce dangereux ?
5. `WHERE` et `HAVING` : lequel filtre les groupes, lequel filtre les lignes ?

*Notez votre score dans la cellule suivante.*

*Votre réponse :*

…

## Pour finir : relevés et livrable

In [ ]:
print("=" * 60)
print("TABLEAU DE RELEVES — TP4")
print("=" * 60)
for k, v in releves.items():
    print(f"{k:32s} : {v}")

manquants = [k for k, v in releves.items() if v is None]
print("\nReleves manquants :", manquants if manquants else "aucun — bravo !")

### Pousser le livrable

Depuis la racine de votre dépôt :
```
git status                       # verifier que data/ n'apparait PAS
git add notebooks/TP4_spark_sql.ipynb
git commit -m "TP4 : requetes SQL et nettoyage FCFA"
git push
```

**Checklist finale**
- [ ] Notebook exécuté de bout en bout (Restart & Run All), sorties visibles ;
- [ ] `nb_null = 0` dans la validation C4 ;
- [ ] Écart CA naïf / nettoyé relevé en FCFA **et** en % ;
- [ ] Une phrase d'interprétation sous chaque indicateur de la partie D ;
- [ ] Aucun relevé manquant dans la cellule ci-dessus ;
- [ ] Données non commitées.

*Séance 5 : les jointures — lecture préalable : Damji et al., Learning Spark 2e éd., chapitre 5.*